In [20]:
import psycopg2
from pgvector.psycopg2 import register_vector
import pandas as pd
import os

def get_medquad_data_from_db(db_name="medquad_db", user="user", password="password", host="localhost", port="5432"):
    """
    Recupera tutti i dati dalla tabella 'medquad' del database PostgreSQL
    e li restituisce come un Pandas DataFrame.

    Args:
        db_name (str): Il nome del database. Default: "medquad_db"
        user (str): Il nome utente per la connessione al database. Default: "user"
        password (str): La password per la connessione al database. Default: "password"
        host (str): L'host del database. Default: "localhost"
        port (str): La porta del database. Default: "5432"

    Returns:
        pd.DataFrame: Un DataFrame Pandas contenente tutti i dati della tabella 'medquad',
                      o None se si verifica un errore.
    """
    conn = None
    try:
        # Connessione al database PostgreSQL
        print(f"Tentativo di connessione al database {db_name} su {host}:{port}...")
        conn = psycopg2.connect(
            dbname=db_name,
            user=user,
            password=password,
            host=host,
            port=port
        )
        # Registra il tipo vettoriale per una corretta gestione da psycopg2
        register_vector(conn)
        print("Connessione al database riuscita.")

        # Esegui la query per recuperare tutti i dati
        query = "SELECT id, question, answer, question_embedding FROM medquad ORDER BY id;"
        print("Esecuzione della query per recuperare i dati...")
        df = pd.read_sql_query(query, conn)
        print(f"Recuperati {len(df)} record dalla tabella 'medquad'.")
        return df

    except Exception as e:
        print(f"Errore durante il recupero dei dati dal database: {e}")
        return None
    finally:
        # Assicurati che la connessione venga chiusa
        if conn:
            conn.close()
            print("Connessione al database chiusa.")

# Nel tuo script principale o in un nuovo script di test:
# from db_utils import get_medquad_data_from_db # Se l'hai salvata in db_utils.py

# Recupera i dati
medquad_df = get_medquad_data_from_db()

if medquad_df is not None:
    print("\nEcco i primi 5 record del DataFrame recuperato:")
    print(medquad_df.head())

    print("\nInformazioni sul DataFrame:")
    medquad_df.info()

    # Esempio di accesso a un elemento specifico:
    # Se vuoi vedere l'embedding della prima domanda:
    if not medquad_df.empty:
        first_embedding = medquad_df.loc[0, 'question_embedding']
        print(f"\nEmbedding della prima domanda (primi 5 valori): {first_embedding[:5]}...")
        print(f"Lunghezza dell'embedding: {len(first_embedding)}")
else:
    print("Impossibile recuperare i dati dal database.")

Tentativo di connessione al database medquad_db su localhost:5432...
Connessione al database riuscita.
Esecuzione della query per recuperare i dati...


/tmp/ipykernel_31140/2191324629.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


Recuperati 47441 record dalla tabella 'medquad'.
Connessione al database chiusa.

Ecco i primi 5 record del DataFrame recuperato:
   id                                           question  \
0   1       What is (are) keratoderma with woolly hair ?   
1   2  How many people are affected by keratoderma wi...   
2   3  What are the genetic changes related to kerato...   
3   4        Is keratoderma with woolly hair inherited ?   
4   5  What are the treatments for keratoderma with w...   

                                              answer  \
0  Keratoderma with woolly hair is a group of rel...   
1  Keratoderma with woolly hair is rare; its prev...   
2  Mutations in the JUP, DSP, DSC2, and KANK2 gen...   
3  Most cases of keratoderma with woolly hair hav...   
4  These resources address the diagnosis or manag...   

                                  question_embedding  
0  [-0.018395241, 0.06388804, 0.030589689, 0.0034...  
1  [0.04390024, 0.037470132, 0.05630666, 0.015431...  
2  [-0.

In [21]:
medquad_df

,id,question,answer,question_embedding
0,1,What is (are) keratoderma with woolly hair ?,Keratoderma with woolly hair is a group of rel...,"[-0.018395241, 0.06388804, 0.030589689, 0.0034..."
1,2,How many people are affected by keratoderma wi...,Keratoderma with woolly hair is rare; its prev...,"[0.04390024, 0.037470132, 0.05630666, 0.015431..."
2,3,What are the genetic changes related to kerato...,"Mutations in the JUP, DSP, DSC2, and KANK2 gen...","[-0.035967115, 0.04879758, 0.029455611, 0.0031..."
3,4,Is keratoderma with woolly hair inherited ?,Most cases of keratoderma with woolly hair hav...,"[-0.016589213, 0.047216047, 0.035442, 0.000691..."
4,5,What are the treatments for keratoderma with w...,These resources address the diagnosis or manag...,"[-0.010475527, 0.06641753, 0.080703445, -0.000..."
...,...,...,...,...
47436,47437,What are the side effects or risks of Hydrocod...,None,"[-0.010809549, -0.0123977605, 0.039565854, 0.0..."
47437,47438,What should I know about storage and disposal ...,None,"[-0.01423758, 0.025414992, 0.04754724, 0.00776..."
47438,47439,What to do in case of emergency or overdose of...,None,"[0.014811984, 0.011623636, 0.041266933, -0.026..."
47439,47440,What other information should I know about Hyd...,None,"[-0.0016767287, -0.04796518, 0.053300414, 0.03..."


In [17]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import psycopg2
from pgvector.psycopg2 import register_vector
import os

ds = load_dataset("lavita/MedQuAD")
data = ds['train']
print(f"# data in MedQuAD: {len(data)}")

# data in MedQuAD: 47441


In [19]:
df = data.to_pandas()

df.head(5)

,document_id,document_source,document_url,category,umls_cui,umls_semantic_types,umls_semantic_group,synonyms,question_id,question_focus,question_type,question,answer
0,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-1,keratoderma with woolly hair,information,What is (are) keratoderma with woolly hair ?,Keratoderma with woolly hair is a group of rel...
1,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-2,keratoderma with woolly hair,frequency,How many people are affected by keratoderma wi...,Keratoderma with woolly hair is rare; its prev...
2,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-3,keratoderma with woolly hair,genetic changes,What are the genetic changes related to kerato...,"Mutations in the JUP, DSP, DSC2, and KANK2 gen..."
3,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-4,keratoderma with woolly hair,inheritance,Is keratoderma with woolly hair inherited ?,Most cases of keratoderma with woolly hair hav...
4,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-5,keratoderma with woolly hair,treatment,What are the treatments for keratoderma with w...,These resources address the diagnosis or manag...
